# Marketing Analytics: Environment & Data Setup

**Run this notebook once** before the marketing analytics demo to prepare the
environment, create the database layout, and generate synthetic marketing data.

After this notebook completes, the demo notebook
(`workspace_marketing_demo.ipynb`) runs from Section 1 with no setup delays.

**What this notebook does:**
1. Install R environment with CausalImpact, Robyn, and their dependencies (~5 min first time)
2. Install snowflakeR and RSnowflake
3. Create a dedicated database with schemas for source data, Feature Store,
   training artefacts, and model registry
4. Generate four synthetic marketing tables (campaigns, ad spend, revenue, holidays)
5. Validate everything is ready

## 1. Install R Environment

In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(config="snowflaker_marketing_config.yaml")

In [ ]:
%%R
library(snowflakeR)
library(CausalImpact)
library(tidyverse)
cat("snowflakeR", as.character(packageVersion("snowflakeR")), "loaded\n")
cat("CausalImpact", as.character(packageVersion("CausalImpact")), "loaded\n")

## 2. Connect and Create Database Layout

Create the database with four isolated schemas:

| Schema | Purpose |
|---|---|
| `RAW_DATA` | Source tables (campaigns, ad spend, revenue, holidays) |
| `FEATURES` | Feature Store (entity, feature views, dynamic tables) |
| `TRAINING` | Materialised training datasets |
| `MODELS` | Model Registry (versioned models, metrics) |

In [ ]:
%%R
conn <- sfr_connect()
conn

In [ ]:
%%R
ma_db       <- "MARKETING_ANALYTICS_ML"
ma_source   <- "RAW_DATA"
ma_features <- "FEATURES"
ma_training <- "TRAINING"
ma_registry <- "MODELS"

fqn_source <- function(name) paste(ma_db, ma_source, name, sep = ".")

cat("Database layout:\n")
cat(sprintf("  Database:       %s\n", ma_db))
cat(sprintf("  Source data:    %s.%s\n", ma_db, ma_source))
cat(sprintf("  Feature Store:  %s.%s\n", ma_db, ma_features))
cat(sprintf("  Training:       %s.%s\n", ma_db, ma_training))
cat(sprintf("  Model Registry: %s.%s\n", ma_db, ma_registry))

### Create Database and Schemas

In [ ]:
%%R
sfr_execute(conn, paste("CREATE DATABASE IF NOT EXISTS", ma_db))

for (schema in c(ma_source, ma_features, ma_training, ma_registry)) {
  sfr_execute(conn, paste(
    "CREATE SCHEMA IF NOT EXISTS", paste(ma_db, schema, sep = ".")
  ))
}

rcat(sprintf("Database %s ready with 4 schemas.", ma_db))

## 3. Generate Synthetic Marketing Data

Four tables simulate a marketing analytics data warehouse. All data is
generated in Snowflake using SQL -- no external files needed.

- **CAMPAIGNS** -- Campaign metadata (~10 campaigns across TV, Facebook, Search)
- **DAILY_AD_SPEND** -- Daily advertising spend, impressions, clicks by channel and market (~10K rows)
- **DAILY_MARKET_REVENUE** -- Daily revenue and conversions by market (~3K rows)
- **HOLIDAY_CALENDAR** -- Canadian statutory holidays for Prophet seasonality

The data spans 2024-01-01 to 2026-02-28 (~2 years). A specific Montreal TV
campaign in Q3 2025 serves as the intervention for CausalImpact analysis.

In [ ]:
%%R
tbl_campaigns <- fqn_source("CAMPAIGNS")
tbl_spend     <- fqn_source("DAILY_AD_SPEND")
tbl_revenue   <- fqn_source("DAILY_MARKET_REVENUE")
tbl_holidays  <- fqn_source("HOLIDAY_CALENDAR")

cat("Tables will be created as:\n")
cat(" ", tbl_campaigns, "\n")
cat(" ", tbl_spend, "\n")
cat(" ", tbl_revenue, "\n")
cat(" ", tbl_holidays, "\n")

### 3.1 Campaign Metadata

Campaigns across three channels (TV, Facebook, Search) and three markets
(Montreal, Toronto, Vancouver). One campaign -- the Montreal Q3 2025 TV push --
is the intervention whose causal effect we will measure.

In [ ]:
%%R
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", tbl_campaigns, "(",
  "  CAMPAIGN_ID INTEGER,",
  "  CAMPAIGN_NAME STRING,",
  "  CHANNEL STRING,",
  "  MARKET STRING,",
  "  START_DATE DATE,",
  "  END_DATE DATE,",
  "  DAILY_BUDGET FLOAT,",
  "  OBJECTIVE STRING",
  ")"
))

sfr_execute(conn, paste(
  "INSERT INTO", tbl_campaigns, "VALUES",
  "(1,  'National TV Branding',    'TV',       'NATIONAL',  '2024-01-01', '2026-02-28', 1500, 'BRAND_AWARENESS'),",
  "(2,  'Montreal TV - Q3 Push',   'TV',       'MONTREAL',  '2025-07-01', '2025-09-30', 4000, 'MARKET_EXPANSION'),",
  "(3,  'Toronto TV - Spring',     'TV',       'TORONTO',   '2026-03-01', '2026-05-31', 2000, 'BRAND_AWARENESS'),",
  "(4,  'Facebook - Always On',    'FACEBOOK', 'NATIONAL',  '2024-01-01', '2026-02-28',  600, 'LEAD_GENERATION'),",
  "(5,  'Facebook - Holiday Push', 'FACEBOOK', 'NATIONAL',  '2025-11-15', '2025-12-31', 1200, 'PERFORMANCE'),",
  "(6,  'Search - Brand Terms',    'SEARCH',   'NATIONAL',  '2024-01-01', '2026-02-28',  400, 'PERFORMANCE'),",
  "(7,  'Search - Competitor',     'SEARCH',   'NATIONAL',  '2024-06-01', '2026-02-28',  500, 'CONQUEST'),",
  "(8,  'Facebook - Montreal Geo', 'FACEBOOK', 'MONTREAL',  '2025-01-01', '2026-02-28',  300, 'LOCAL_AWARENESS'),",
  "(9,  'Facebook - Toronto Geo',  'FACEBOOK', 'TORONTO',   '2025-01-01', '2026-02-28',  350, 'LOCAL_AWARENESS'),",
  "(10, 'Facebook - Van Geo',      'FACEBOOK', 'VANCOUVER', '2025-01-01', '2026-02-28',  250, 'LOCAL_AWARENESS')"
))

rcat(sprintf("Created %s with 10 campaigns", tbl_campaigns))
sfr_query(conn, paste("SELECT channel, COUNT(*) AS n FROM", tbl_campaigns, "GROUP BY channel"))

### 3.2 Daily Advertising Spend

Three years of daily advertising data across TV, Facebook, and Search
channels, broken out by market (Montreal, Toronto, Vancouver). Spend
levels vary by channel, market size, seasonality, and day-of-week. The
Montreal TV campaign in Q3 2025 shows a visible spend increase.

In [ ]:
%%R
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", tbl_spend, "AS",
  "WITH date_spine AS (",
  "  SELECT DATEADD('day', ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,",
  "    '2024-01-01'::DATE) AS dt",
  "  FROM TABLE(GENERATOR(ROWCOUNT => 790))",
  "),",
  "channel_market AS (",
  "  SELECT c.column1 AS channel, m.column1 AS market",
  "  FROM (VALUES ('TV'), ('FACEBOOK'), ('SEARCH')) AS c,",
  "       (VALUES ('MONTREAL'), ('TORONTO'), ('VANCOUVER')) AS m",
  "),",
  "spend_base AS (",
  "  SELECT d.dt, cm.channel, cm.market,",
  "    ROUND(GREATEST(50,",
  "      CASE cm.channel",
  "        WHEN 'TV' THEN 1500 WHEN 'FACEBOOK' THEN 600 ELSE 800",
  "      END",
  "      * CASE cm.market",
  "          WHEN 'MONTREAL' THEN 1.3 WHEN 'TORONTO' THEN 1.1 ELSE 0.8",
  "        END",
  "      * (1 + 0.15 * SIN(2 * PI() * (DAYOFYEAR(d.dt) - 80) / 365.0))",
  "      * CASE WHEN DAYOFWEEK(d.dt) IN (0, 6) AND cm.channel != 'TV'",
  "             THEN 0.7 ELSE 1.0 END",
  "      * (1 + UNIFORM(-20, 20, RANDOM()) / 100.0)",
  "      * CASE WHEN cm.channel = 'TV' AND cm.market = 'MONTREAL'",
  "             AND d.dt BETWEEN '2025-07-01' AND '2025-09-30'",
  "             THEN 2.5 ELSE 1.0 END",
  "    ), 2) AS spend",
  "  FROM date_spine d CROSS JOIN channel_market cm",
  "),",
  "with_impressions AS (",
  "  SELECT *,",
  "    CASE",
  "      WHEN channel = 'FACEBOOK'",
  "        THEN ROUND(spend * UNIFORM(80, 120, RANDOM()))",
  "      WHEN channel = 'SEARCH'",
  "        THEN ROUND(spend * UNIFORM(5, 15, RANDOM()))",
  "      ELSE NULL",
  "    END AS impressions",
  "  FROM spend_base",
  ")",
  "SELECT dt AS DATE, channel, market, spend, impressions,",
  "  CASE",
  "    WHEN channel = 'FACEBOOK'",
  "      THEN ROUND(GREATEST(0, impressions * UNIFORM(1, 5, RANDOM()) / 100.0))",
  "    WHEN channel = 'SEARCH'",
  "      THEN ROUND(GREATEST(0, impressions * UNIFORM(3, 12, RANDOM()) / 100.0))",
  "    ELSE NULL",
  "  END AS clicks",
  "FROM with_impressions"
))

n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl_spend))
rcat(sprintf("Created %s: %s rows", tbl_spend, format(n$N, big.mark = ",")))

sfr_query(conn, paste(
  "SELECT channel, market, COUNT(*) AS days,",
  "  ROUND(AVG(spend), 0) AS avg_daily_spend",
  "FROM", tbl_spend,
  "GROUP BY channel, market",
  "ORDER BY channel, market"
))

### 3.3 Daily Market Revenue

Daily revenue, conversions, and website sessions for each market.
Revenue shares the same seasonal patterns as ad spend and includes
a detectable uplift in Montreal during the Q3 2025 TV campaign --
this is the causal effect that CausalImpact will estimate.

In [ ]:
%%R
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", tbl_revenue, "AS",
  "WITH date_spine AS (",
  "  SELECT DATEADD('day', ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1,",
  "    '2024-01-01'::DATE) AS dt",
  "  FROM TABLE(GENERATOR(ROWCOUNT => 790))",
  "),",
  "markets AS (",
  "  SELECT column1 AS market",
  "  FROM VALUES ('MONTREAL'), ('TORONTO'), ('VANCOUVER')",
  "),",
  "revenue_base AS (",
  "  SELECT d.dt, m.market,",
  "    ROUND(GREATEST(1000,",
  "      CASE m.market",
  "        WHEN 'MONTREAL' THEN 12000",
  "        WHEN 'TORONTO' THEN 14000",
  "        ELSE 9000",
  "      END",
  "      * (1 + 0.20 * SIN(2 * PI() * (DAYOFYEAR(d.dt) - 80) / 365.0))",
  "      * CASE DAYOFWEEK(d.dt) WHEN 0 THEN 0.85 WHEN 6 THEN 0.90 ELSE 1.0 END",
  "      + UNIFORM(-1500, 1500, RANDOM())",
  "      + CASE WHEN m.market = 'MONTREAL'",
  "             AND d.dt BETWEEN '2025-07-01' AND '2025-09-30'",
  "             THEN 2000 ELSE 0 END",
  "    ), 2) AS revenue",
  "  FROM date_spine d CROSS JOIN markets m",
  "),",
  "with_conversions AS (",
  "  SELECT *,",
  "    ROUND(GREATEST(10, revenue / UNIFORM(40, 60, RANDOM()))) AS conversions",
  "  FROM revenue_base",
  ")",
  "SELECT dt AS DATE, market, revenue, conversions,",
  "  ROUND(GREATEST(100, conversions * UNIFORM(15, 25, RANDOM()))) AS website_sessions",
  "FROM with_conversions"
))

n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl_revenue))
rcat(sprintf("Created %s: %s rows", tbl_revenue, format(n$N, big.mark = ",")))

sfr_query(conn, paste(
  "SELECT market, ROUND(AVG(revenue), 0) AS avg_daily_revenue,",
  "  ROUND(AVG(CASE WHEN DATE BETWEEN '2025-07-01' AND '2025-09-30'",
  "    THEN revenue END), 0) AS avg_revenue_q3_2025",
  "FROM", tbl_revenue,
  "GROUP BY market"
))

### 3.4 Holiday Calendar

Canadian statutory holidays for use with Prophet seasonality decomposition
in Robyn. Formatted with `DS` (date), `HOLIDAY` (name), and `COUNTRY`
columns matching Prophet's expected schema.

In [ ]:
%%R
sfr_execute(conn, paste(
  "CREATE OR REPLACE TABLE", tbl_holidays, "(",
  "  DS DATE, HOLIDAY STRING, COUNTRY STRING",
  ")"
))

sfr_execute(conn, paste(
  "INSERT INTO", tbl_holidays, "VALUES",
  "('2024-01-01','NewYear','CA'), ('2024-03-29','GoodFriday','CA'),",
  "('2024-05-20','VictoriaDay','CA'), ('2024-06-24','SaintJeanBaptiste','CA'),",
  "('2024-07-01','CanadaDay','CA'), ('2024-09-02','LabourDay','CA'),",
  "('2024-09-30','TruthReconciliation','CA'), ('2024-10-14','Thanksgiving','CA'),",
  "('2024-12-25','Christmas','CA'), ('2024-12-26','BoxingDay','CA'),",
  "('2025-01-01','NewYear','CA'), ('2025-04-18','GoodFriday','CA'),",
  "('2025-05-19','VictoriaDay','CA'), ('2025-06-24','SaintJeanBaptiste','CA'),",
  "('2025-07-01','CanadaDay','CA'), ('2025-09-01','LabourDay','CA'),",
  "('2025-09-30','TruthReconciliation','CA'), ('2025-10-13','Thanksgiving','CA'),",
  "('2025-12-25','Christmas','CA'), ('2025-12-26','BoxingDay','CA'),",
  "('2026-01-01','NewYear','CA'), ('2026-04-03','GoodFriday','CA'),",
  "('2026-05-18','VictoriaDay','CA'), ('2026-06-24','SaintJeanBaptiste','CA'),",
  "('2026-07-01','CanadaDay','CA'), ('2026-09-07','LabourDay','CA'),",
  "('2026-09-30','TruthReconciliation','CA'), ('2026-10-12','Thanksgiving','CA'),",
  "('2026-12-25','Christmas','CA'), ('2026-12-26','BoxingDay','CA')"
))

n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl_holidays))
rcat(sprintf("Created %s: %s holidays", tbl_holidays, n$N))

## 4. Validation

Confirm the database, schemas, and source tables are ready for the demo.

In [ ]:
%%R
rcat("=== Setup Validation ===")
rcat("")

rcat(sprintf("Database: %s", ma_db))
rcat("Schemas:")
for (s in c(ma_source, ma_features, ma_training, ma_registry)) {
  rcat(sprintf("  %s.%s", ma_db, s))
}
rcat("")

rcat("Tables:")
for (tbl in c(tbl_campaigns, tbl_spend, tbl_revenue, tbl_holidays)) {
  n <- sfr_query(conn, paste("SELECT COUNT(*) AS n FROM", tbl))
  rcat(sprintf("  %s: %s rows", tbl, format(n$N, big.mark = ",")))
}

rcat("")
rcat(sprintf("Date range: %s",
  sfr_query(conn, paste(
    "SELECT MIN(DATE) || ' to ' || MAX(DATE) AS range FROM", tbl_spend
  ))$RANGE
))

rcat("")
rcat("Intervention check (Montreal Q3 2025 avg revenue vs baseline):")
sfr_query(conn, paste(
  "SELECT market,",
  "  ROUND(AVG(CASE WHEN DATE < '2025-07-01' THEN revenue END), 0) AS baseline,",
  "  ROUND(AVG(CASE WHEN DATE BETWEEN '2025-07-01' AND '2025-09-30'",
  "    THEN revenue END), 0) AS intervention_period,",
  "  ROUND(AVG(CASE WHEN DATE BETWEEN '2025-07-01' AND '2025-09-30'",
  "    THEN revenue END) - AVG(CASE WHEN DATE < '2025-07-01'",
  "    THEN revenue END), 0) AS lift",
  "FROM", tbl_revenue,
  "GROUP BY market ORDER BY market"
))

rcat("")
rcat("Setup complete. You can now run workspace_marketing_demo.ipynb.")

## 5. Cleanup (Optional)

Uncomment to tear down everything created by this setup notebook.
**Warning:** This drops the entire database including all schemas, tables,
Feature Views, and registered models.

In [ ]:
%%R
# sfr_execute(conn, paste("DROP DATABASE IF EXISTS", ma_db))
# rcat(sprintf("Dropped database %s", ma_db))

cat("Cleanup section -- uncomment the line above to drop everything\n")

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.MARKETING_SETUP_RESULTS AS
        SELECT 'marketing_setup' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.MARKETING_SETUP_RESULTS")
except Exception:
    pass